# 09 · Baseline models and the unseen-perturbation benchmark
Task: given a perturbation's gene features, predict its delta vector. Splits are at the **perturbation level** (test genes never appear in training), repeated over 5 seeds.

Models: control (zero change), mean delta, Ridge, kNN (similar genes → similar effects), Random Forest.

In [ ]:
# ---- Bootstrap: Colab (Google Drive) or local ----
import os, sys, subprocess
REPO_URL = "https://github.com/<your-username>/PerturbAI.git"   # <-- edit once after you create the repo
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/PerturbAI"):
        subprocess.run(["git", "clone", REPO_URL, "/content/PerturbAI"], check=True)
    os.chdir("/content/PerturbAI")
    subprocess.run(["git", "pull", "-q"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
else:
    try:
        import perturbai  # installed with `pip install -e .`
    except ImportError:
        sys.path.insert(0, os.path.abspath("../src"))  # fallback when running from notebooks/
from perturbai.paths import get_paths
P = get_paths()
print("Data root:", P.root)

In [ ]:
import numpy as np, pandas as pd, scipy.sparse as sp, matplotlib.pyplot as plt
from perturbai.paths import artifact
from perturbai.utils import set_seed, load_config
CFG = load_config("data_config.yaml"); MCFG = load_config("model_config.yaml")
set_seed(CFG["seed"])
SYNTHETIC = False        # True = 1-minute smoke test on simulated data (no download)
DS = "replogle_k562_essential"
INFO = CFG["datasets"][DS]
PERT, CTRL, BATCH = INFO["perturbation_col"], INFO["control_label"], INFO["batch_col"]
TAG = DS + ("_synthetic" if SYNTHETIC else "")   # artifact prefix

In [ ]:
from perturbai.pipeline import load_task
from perturbai.evaluation.benchmark import run_benchmark, summarize, compare_to_baseline
from perturbai.models.baselines import ControlBaseline, MeanBaseline, RidgeModel, KNNModel, RandomForestModel
T = load_task(P, TAG, ("coexp_",))
print("task:", T["X"].shape, T["Y"].shape, "| perturbations without features:", len(T["missing"]))
M = MCFG["models"]
factories = {"control": ControlBaseline, "mean": MeanBaseline,
             "ridge": lambda: RidgeModel(**M["ridge"]), "knn": lambda: KNNModel(**M["knn"]),
             "rf": lambda: RandomForestModel(M["rf"]["n_estimators"], M["rf"]["n_pca_targets"], M["rf"]["min_samples_leaf"])}
long = run_benchmark(factories, T["X"], T["Y"], T["index"], T["ctrl_mean"], seeds=MCFG["split"]["seeds"],
                     test_frac=MCFG["split"]["test_frac"], top_k=MCFG["metrics"]["top_k"])
long.to_csv(P.results / f"{TAG}_bench_baselines.csv", index=False)

In [ ]:
summ = summarize(long); display(summ.round(3))
display(compare_to_baseline(long, "mean", "pearson_delta").round(3))

In [ ]:
import seaborn as sns
fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
for a, m in zip(ax, ["pearson_delta", "mse_delta", "direction_acc_top_deg"]):
    sns.boxplot(data=long, x="model", y=m, ax=a, showfliers=False); a.set_title(m)
plt.tight_layout(); plt.show()
print("Note: pearson_expr (raw-expression correlation) is near 1 for every model -> uninformative:")
print(long.groupby("model").pearson_expr.mean().round(3))

**Interpretation questions:** Does the mean baseline beat the zero-change baseline (does a *shared* response exist, e.g. general stress)? Do feature-based models beat the mean on *delta* correlation? On which metric do they fail? Compare against the noise ceiling from notebook 05.